# BraTS 2021 — Exploracija podataka

Ovaj notebook učitava MRI snimke jednog pacijenta i prikazuje što se nalazi u svakoj datoteci.

## 1. Uvoz biblioteka i postavljanje putanja

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

%matplotlib inline

DATA_DIR = Path("../data")
MODALITIES = ["flair", "t1", "t1ce", "t2"]
PATIENT_ID = "BraTS2021_00495"

print("Sve biblioteke uspjesno ucitane.")

## 2. Ucitavanje jednog pacijenta

Svaki pacijent ima **5 datoteka** — 4 MRI modaliteta + segmentacijska maska.

In [ ]:
patient_dir = DATA_DIR / PATIENT_ID

print(f"Datoteke za pacijenta {PATIENT_ID}:")
for f in sorted(patient_dir.iterdir()):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name:<45s}  {size_mb:.1f} MB")

In [ ]:
# Ucitaj sve 4 modaliteta
volumes = {}
for mod in MODALITIES:
    path = patient_dir / f"{PATIENT_ID}_{mod}.nii.gz"
    img  = nib.load(str(path))
    vol  = img.get_fdata()
    volumes[mod] = vol
    print(f"  {mod:6s}  shape={vol.shape}  dtype={vol.dtype}  min={vol.min():.0f}  max={vol.max():.0f}  mean={vol.mean():.1f}")

# Ucitaj segmentacijsku masku
seg = nib.load(str(patient_dir / f"{PATIENT_ID}_seg.nii.gz")).get_fdata().astype(np.int32)
print(f"\n  {'seg':6s}  shape={seg.shape}  unique labels={np.unique(seg).tolist()}")

## 3. Sto znaci shape (240, 240, 155)?

Volumen je **3D** — nije samo jedna slika nego stog slika:
- 240 x 240 piksela po svakom rezu (axial slice)
- 155 rezova (slices) po osi Z

Ukupno: **240 × 240 × 155 = 8,928,000 voksela** po modalitetu.

In [ ]:
flair = volumes["flair"]
print(f"Dimenzije: {flair.shape}")
print(f"  osa X (lijevo-desno):       {flair.shape[0]} piksela")
print(f"  osa Y (prednje-zadnje):     {flair.shape[1]} piksela")
print(f"  osa Z (broj axial sliceova): {flair.shape[2]} rezova")
print(f"\nUkupno voksela po modalitetu: {flair.size:,}")
print(f"Ukupno voksela (sva 4 modal.): {flair.size * 4:,}")

## 4. Prikaz jednog reza — sva 4 modaliteta + segmentacija

In [ ]:
# Odaberi sredisnji rez (z = 77 od 155)
z = seg.shape[2] // 2

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f"{PATIENT_ID}  |  axial slice z={z}  (od ukupno {seg.shape[2]})", fontsize=13)

titles = ["FLAIR\n(edem)", "T1\n(anatomija)", "T1ce\n(aktivna jezgra)", "T2\n(tekucina)", "Segmentacija"]

for i, mod in enumerate(MODALITIES):
    axes[i].imshow(volumes[mod][:, :, z].T, cmap="gray", origin="lower")
    axes[i].set_title(titles[i])
    axes[i].axis("off")

axes[4].imshow(seg[:, :, z].T, cmap="tab10", vmin=0, vmax=4, origin="lower")
axes[4].set_title(titles[4])
axes[4].axis("off")

# Legenda za segmentaciju
colors = plt.cm.tab10(np.linspace(0, 1, 10))
legend = [
    mpatches.Patch(color=colors[0], label="0 - Zdravo tkivo"),
    mpatches.Patch(color=colors[1], label="1 - Nekroticna jezgra"),
    mpatches.Patch(color=colors[2], label="2 - Edem"),
    mpatches.Patch(color=colors[4], label="4 - Aktivni tumor (ET)"),
]
axes[4].legend(handles=legend, loc="lower left", fontsize=7, framealpha=0.8)

plt.tight_layout()
plt.savefig("../outputs/sample_slice.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Prikaz vise rezova — vidimo 3D volumen

In [ ]:
# Prikazi 8 razlicitih rezova FLAIR modaliteta
slices = np.linspace(20, 130, 8, dtype=int)

fig, axes = plt.subplots(2, 8, figsize=(22, 6))
fig.suptitle("FLAIR i Segmentacija — razliciti axial rezovi", fontsize=13)

for i, z in enumerate(slices):
    axes[0, i].imshow(volumes["flair"][:, :, z].T, cmap="gray", origin="lower")
    axes[0, i].set_title(f"z={z}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(seg[:, :, z].T, cmap="tab10", vmin=0, vmax=4, origin="lower")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("FLAIR", fontsize=10)
axes[1, 0].set_ylabel("Seg", fontsize=10)

plt.tight_layout()
plt.show()

## 6. Raspodjela klasa — neuravnotezenost

Ovo je kljucni problem: tumor zauzima samo **~1% voksela**.

In [ ]:
total = seg.size
label_info = [
    (0, "Zdravo tkivo (background)"),
    (1, "Nekroticna jezgra (NCR)"),
    (2, "Edem (ED)"),
    (4, "Aktivni tumor (ET)"),
]

print(f"{'Label':<6} {'Naziv':<28} {'Voksela':>10} {'Postotak':>9}")
print("-" * 58)
counts = []
for label, name in label_info:
    count = int((seg == label).sum())
    counts.append(count)
    print(f"  {label:<4} {name:<28} {count:>10,} {100*count/total:>8.2f}%")

# Graficki prikaz
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

names  = [info[1] for info in label_info]
colors_bar = ["steelblue", "tomato", "goldenrod", "mediumseagreen"]

# Pie chart sa svim klasama
ax1.pie(counts, labels=[f"L{l}" for l, _ in label_info],
        colors=colors_bar, autopct="%1.2f%%", startangle=140)
ax1.set_title("Raspodjela svih klasa")

# Bar chart samo tumorskih klasa (bez backgrounda)
tumor_counts = counts[1:]
tumor_names  = ["NCR (label 1)", "ED (label 2)", "ET (label 4)"]
bars = ax2.bar(tumor_names, tumor_counts, color=colors_bar[1:])
ax2.set_title("Tumorske klase (bez backgrounda)")
ax2.set_ylabel("Broj voksela")
for bar, val in zip(bars, tumor_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f"{val:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. BraTS evaluacijske regije

BraTS ne evaluira direktno 4 klase nego **3 izvedene regije**:

| Regija | Definicija | Klinicki smisao |
|--------|-----------|------------------|
| **WT** (Whole Tumor) | labele 1 + 2 + 4 | Cijeli tumor |
| **TC** (Tumor Core)  | labele 1 + 4     | Jezgra bez edema |
| **ET** (Enhancing Tumor) | labela 4 only | Aktivni dio |

In [ ]:
WT = (seg > 0).astype(np.uint8)                         # 1+2+4
TC = ((seg == 1) | (seg == 4)).astype(np.uint8)         # 1+4
ET = (seg == 4).astype(np.uint8)                        # 4

print(f"{'Regija':<22} {'Voksela':>10} {'Postotak':>9}")
print("-" * 44)
for name, mask in [("WT (Whole Tumor)", WT), ("TC (Tumor Core)", TC), ("ET (Enhancing)", ET)]:
    print(f"  {name:<20} {mask.sum():>10,} {100*mask.sum()/total:>8.3f}%")

# Prikaz na jednom rezu
z = seg.shape[2] // 2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f"BraTS evaluacijske regije  |  slice z={z}", fontsize=13)

axes[0].imshow(volumes["flair"][:, :, z].T, cmap="gray", origin="lower")
axes[0].set_title("FLAIR (referenca)")

axes[1].imshow(WT[:, :, z].T, cmap="Greens", origin="lower")
axes[1].set_title("WT — Whole Tumor\n(labele 1+2+4)")

axes[2].imshow(TC[:, :, z].T, cmap="Oranges", origin="lower")
axes[2].set_title("TC — Tumor Core\n(labele 1+4)")

axes[3].imshow(ET[:, :, z].T, cmap="Reds", origin="lower")
axes[3].set_title("ET — Enhancing Tumor\n(labela 4)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Raspodjela intenziteta po modalitetima

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Raspodjela intenziteta po modalitetu (samo mozak, bez pozadine)", fontsize=12)

brain_mask = (volumes["flair"] > 0)  # gruba maska mozga

for i, mod in enumerate(MODALITIES):
    vals = volumes[mod][brain_mask].flatten()
    axes[i].hist(vals, bins=80, color="steelblue", edgecolor="none", alpha=0.8)
    axes[i].set_title(f"{mod.upper()}\nmax={vals.max():.0f}, mean={vals.mean():.0f}")
    axes[i].set_xlabel("Intenzitet")
    axes[i].set_ylabel("Broj voksela")

plt.tight_layout()
plt.show()

print("Napomena: intenziteti su u razlicitim rasponima po modalitetima.")
print("Normalizacija (Z-score ili min-max) je obavezna prije treniranja!")

## 9. Provjera drugog pacijenta

In [ ]:
p2_id = "BraTS2021_00621"
p2_dir = DATA_DIR / p2_id

print(f"Provjera pacijenta {p2_id}:")
for mod in MODALITIES + ["seg"]:
    vol = nib.load(str(p2_dir / f"{p2_id}_{mod}.nii.gz")).get_fdata()
    print(f"  {mod:6s}  shape={vol.shape}")

seg2 = nib.load(str(p2_dir / f"{p2_id}_seg.nii.gz")).get_fdata().astype(int)
print(f"\nLabele u segmentaciji: {np.unique(seg2).tolist()}")
print("\nOba pacijenta imaju isti format.")

## Zakljucak

**Sto smo naucili:**

1. Svaki pacijent ima 4 MRI modaliteta + segmentacijsku masku, svi oblika **(240, 240, 155)**
2. Labele su `0, 1, 2, 4` — nema labele 3 (remap na 0,1,2,3 potreban za trening)
3. Ekstremna neuravnotezenost: **98.89% pozadina, samo ~1.1% tumor**
4. Evaluacija se radi po 3 regije: WT, TC, ET — ne direktno po 4 klase
5. Normalizacija intenziteta je obavezna jer su rasponi razliciti po modalitetima

**Sljedeci korak:** Preprocessing — normalizacija + remap labela + patch sampling